In [ ]:
%pip install -r requirements.txt

In [ ]:
#lets define a function to connect to SQLDB

import os
from dotenv import load_dotenv
import pyodbc
import struct
from azure.identity import DefaultAzureCredential



def get_mssql_connection():
    # Retrieve the connection string from the environment variables
    #entra_connection_string = os.getenv('ENTRAID_CONNECTION_STRING')
    sql_connection_string = os.getenv('SQL_CONNECTION_STRING')

    # Determine the authentication method and connect to the database
    if sql_connection_string:
        # SQL Authentication
        print("Connecting using SQL Authentication")        
        conn = pyodbc.connect(sql_connection_string)
    else:
        raise ValueError("No valid connection string found in the environment variables.")

    return conn

In [4]:
import os
import requests
import sys
from num2words import num2words
import pandas as pd
import numpy as np
from openai import AzureOpenAI

# Get info from the environment variables
openai_embedding_model = os.environ.get('AZURE_OPENAI_EMBEDDING_MODEL_DEPLOYMENT_NAME')
openai_url = os.environ.get('AZURE_OPENAI_ENDPOINT') + f"/openai/deployments/{openai_embedding_model}/embeddings?api-version=2023-03-15-preview"
openai_key = os.environ.get('AZURE_OPENAI_API_KEY')

def get_embedding(text):
    """
    Get sentence embedding using the Azure OpenAI text-embedding-small model.

    Args:
        text (str): Text to embed.

    Returns:
        dict: A dictionary containing the embedding.
    """
    response = requests.post(openai_url,
        headers={"api-key": openai_key, "Content-Type": "application/json"},
        json={"input": [text]}  # Embed a single sentence
    )
    embedding = response.json()['data'][0]['embedding']
    return embedding

In [5]:
import os
import json
from dotenv import load_dotenv
import pyodbc
import struct
from azure.identity import DefaultAzureCredential

def vector_search_sql(query, num_results=5):
    # Load environment variables from .env file
    load_dotenv()

    #Use the get_mssql_connection function to get the connection string details
    conn = get_mssql_connection()

    # Create a cursor object
    cursor = conn.cursor()

    # Generate the query embedding for the user's search query
    user_query_embedding = get_embedding(query)

    # Convert user_query_embedding to a JSON string
    user_query_embedding_json = json.dumps(user_query_embedding)

    # SQL query for similarity search using the function vector_distance to calculate cosine similarity
    sql_similarity_search = """
    SELECT TOP(?) ProductId, Summary, text,
           1 - vector_distance('cosine', cast(? as vector(1536)), [Embedding]) AS similarity_score
    FROM dbo.ReviewsFood
    ORDER BY similarity_score desc
    """

    cursor.execute(sql_similarity_search, (num_results, user_query_embedding_json))
    results = cursor.fetchall()

    # Close the database connection
    conn.close()

    return results


<small>

# Performing Vector Similarity Search in Azure SQL Database

In this guide, we demonstrate how to query an embedding table in Azure SQL Database to retrieve the most relevant customer reviews based on a user's search query.

## Overview

When a user enters a search query, we convert that text into its **vector representation** using embeddings. This vector can then be compared to the vectors of existing customer reviews stored in the database.

By calculating the **cosine similarity** between the query vector and each review vector, we can determine which reviews are most relevant to the user's intent. These are the reviews most likely associated with the product or experience the user is looking for.

The most relevant reviews—those with the highest similarity—help users discover products or experiences that match their interests.

## Vector Distance in SQL

Azure SQL DB provides built-in support for calculating vector distances. The function syntax is:

```sql
VECTOR_DISTANCE('distance metric', vector1, vector2)


In [6]:
# Assuming you have implemented the `vector_search_sql` function as shown earlier

# Example usage
query = "Healthy food for weight loss"
num_results = 4
search_results = vector_search_sql(query, num_results)
for result in search_results:
    product_id = result[0]  # Assuming ProductId is the first column
    summary = result [1]
    text = result [2]
    similarity_score = result[3]  # Assuming similarity_score is the third column
    
    print(f"Product ID: {product_id}")
    print(f"summary : {summary}")
    print (f"Text : {text}")
    print(f"Similarity Score: {similarity_score}\n")


Connecting using SQL Authentication
Product ID: B003EML8PM
summary : Love it
Text : These are wonderful and they are approvesd snacks for the weight loss program I am on.
Similarity Score: 0.47677781438779876

Product ID: B008ZRKZSM
summary : Delicious minus the calories/fat
Text : This product is a delicious, satisfying alternative for anyone who is trying to cut calories and fat (or on Weight Watchers), without losing great taste.  It's easy to make and my kids even love it!  Great eaten alone or with celery. I've even used it with banana and fat free vanilla yogurt for a smoothie and it was excellent! Highly recommend!
Similarity Score: 0.46474031295756646

Product ID: B001EO5U3I
summary : Nutritious Oat Meal
Text : The product is very nutritious and it is very tasty food for breaksfast and best food for Diet Control.
Similarity Score: 0.45735796259718364

Product ID: B000FDKUSO
summary : best snack
Text : Susie's thin cakes are a great snack if you are trying to lose weight or just

### 🔍 Azure OpenAI Chat Completion with Vector Search Results

This cell sets up a function that uses Azure OpenAI's GPT-4 model to generate intelligent and humorous responses based on product-related search results retrieved from a vector database.

**Key Steps:**
1. **Environment Setup**: Loads API credentials from a `.env` file.
2. **Client Initialization**: Connects to Azure OpenAI using the `AzureOpenAI` client.
3. **Vector Search**: Calls a function (`vector_search_sql`) to retrieve relevant product data based on the user's query.
4. **Message Construction**: Formats the search results and user input into a structured message list.
5. **Chat Completion**: Sends the messages to the GPT-4 model deployed on Azure and returns the AI-generated response.

> 💡 The assistant is instructed to only use the provided search results and include a fun fact in its response.


In [7]:
import os
from dotenv import load_dotenv
from openai import AzureOpenAI

# Load environment variables from .env file
load_dotenv()

# Retrieve the API key and endpoint from the environment variables
api_key = os.getenv('AZURE_OPENAI_API_KEY')
azure_endpoint = os.getenv('AZURE_OPENAI_ENDPOINT')

# Create a chat completion request
client = AzureOpenAI(
    api_key=api_key,
    api_version="2023-05-15",
    azure_endpoint=azure_endpoint
)


def generate_completion(search_results, user_input):
    system_prompt = '''
You are an intelligent & funny assistant who will exclusively answer based on the data provided in the `search_results`:
- Use the information from `search_results` to generate your responses. If the data is not a perfect match for the user's query, use your best judgment to provide helpful suggestions and include the following format:
  Product ID: {product_id}
  Summary: {summary}
  Review: {text}
  Similarity Score: {similarity_score}
- Avoid any other external data sources.
- Add a fun fact related to the overall product searched at the end of the recommendations.
'''

    messages = [{"role": "system", "content": system_prompt}]
    search_results = vector_search_sql(user_input, num_results)
    
    # Create an empty list to store the results
    result_list = []

    # Iterate through the search results and append relevant information to the list
    for result in search_results:
        product_id = result[0]  # Assuming ProductId is the first column
        summary = result[1]
        text = result[2]
        similarity_score = result[3]  # Assuming similarity_score is the third column
        
        # Append the relevant information as a dictionary to the result_list
        result_list.append({
            "product_id": product_id,
            "summary": summary,
            "text": text,
            "similarity_score": similarity_score
        })

    #print (result_list)
    messages.append({"role": "system", "content": f"{result_list}"})
    messages.append({"role": "user", "content": user_input})
    response = client.chat.completions.create(model='gpt-4', messages=messages, temperature=0) #replace with your model deployment name

    return response.dict()

In [9]:
# Create a loop of user input and model output to perform Q&A on the FineFoods Sample data

print("*** What products are you looking for? Ask me & I can help you :) Type 'end' to end the session.\n")

while True:
    user_input = input("User prompt: ")
    if user_input.lower() == "end":
        break

    # Print the user's question
    print(f"\nUser asked: {user_input}")

    # Assuming vector_search_sql and generate_completion are defined functions that work correctly
    search_results = vector_search_sql(user_input)
    completions_results = generate_completion(search_results, user_input)

    # Print the model's response
    print("\nAI's response:")
    print(completions_results['choices'][0]['message']['content'])

# The loop will continue until the user types 'end'


*** What products are you looking for? Ask me & I can help you :) Type 'end' to end the session.


User asked: eat healthy food
Connecting using SQL Authentication
Connecting using SQL Authentication


C:\Users\hetarra\AppData\Local\Temp\ipykernel_100\3720898012.py:58: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.11/migration/
  return response.dict()



AI's response:
Sure, here are some healthy food products you might like:

1. Product ID: B001E6KBSK
   Summary: Great Taste - Healthy Cereal
   Review: Eating healthy never tasted so good. Tough to find in my local stores but Amazon has it at a great price especially through subscribe and save.
   Similarity Score: 0.435196708573105

2. Product ID: B000G6MBX2
   Summary: Plocky's tortilla chips--tasty and healthy
   Review: These chips are the only ones I found to be tasty and healthy. They have fewer fat calories plus higher fiber for those who want good taste and nutrition--the perfect blend!
   Similarity Score: 0.40351376005287576

3. Product ID: B000E15DFM
   Summary: highly recommended
   Review: healthy, delicious, and great for food allergies! no soy, corn etc. Only contains wheat, safflower oil, and salt.
   Similarity Score: 0.4008691457523197

4. Product ID: B008ZRKZSM
   Summary: love this!!
   Review: low fat, low carb, low sugar, full taste! I love this product and recom